# Metadata-reclassification sensitivity probe

Robustness check for the R2 finding ("AI public dialogue is distinctively focused on data and informational autonomy"). The concern is that some reports labelled `AI` in `data/tech_metadata.xlsx` are actually dialogue about data governance / data stewardship / digital identity, and that labelling them `AI` could make R2 partly circular.

This notebook loads the pipeline artefacts and recomputes R1, R2 and R3 under two metadata versions:

- `original` — `data/tech_metadata.xlsx` (41 AI, 11 technology categories)
- `data_split` — `data/tech_metadata_data_split.xlsx` (30 AI + 11 Data = 12 technology categories; data-focused reports relabelled)

No LLM calls. No re-clustering. No re-embedding. Only the metric functions re-run, so this completes in seconds once the embeddings are loaded.

**Reading guide**: if the Privacy & Data Security pp-gap on R2 shrinks substantially under `data_split`, the R2 finding has a labelling sensitivity worth reporting in the paper. If it holds, the finding was never really circular.

In [ ]:
# @title Load pre-computed artefacts
from pathlib import Path
import json as _json
import copy as _copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy as _entropy

from pub_dialogue.utils import AccessStage, AddressStage
from pub_dialogue import multiverse as mv

_access = AccessStage()
_address = AddressStage(access=_access)
OUTPUT_FOLDER = _access.output_folder
CHECKPOINT_FOLDER = _access.checkpoint_folder

# Full artefact bundle (chunks_df + concerns/benefits/embeddings/centroids/mixture weights)
_art = _access.load_artifacts()
chunks_df = _art["chunks_df"]
print(f"Loaded artefacts: {len(chunks_df):,} chunks")

## Load both metadata versions

In [ ]:
# @title Load the two metadata versions and show the technology counts side by side
ORIGINAL_META = pd.read_excel("data/tech_metadata.xlsx")
ALT_META = pd.read_excel("data/tech_metadata_data_split.xlsx")

_counts = pd.concat([
    ORIGINAL_META["technology"].value_counts().rename("original"),
    ALT_META["technology"].value_counts().rename("data_split"),
], axis=1).fillna(0).astype(int)
print(_counts.to_string())
print()

# Report what moved
_merged = ORIGINAL_META.merge(ALT_META, on="filename", suffixes=("_orig", "_alt"))
_moved = _merged[_merged["technology_orig"] != _merged["technology_alt"]]
print(f"\n{len(_moved)} reports reclassified:")
print(_moved[["filename", "technology_orig", "technology_alt"]].to_string(index=False))

## Helper: relabel concerns_df under a given metadata version

The pipeline's LLM extraction, embedding, clustering, labelling and lens-generation are all per-phrase and independent of the technology label. So to swap metadata we only need to re-join `technology_meta` onto `concerns_df` from the alternative file.

In [ ]:
# @title Define relabel_concerns and recompute_cluster_entropy helpers
def relabel_concerns(concerns_df, metadata_df, chunks_df):
    """Return concerns_df with technology_meta re-joined per the given metadata."""
    # Build chunk_id -> technology using source_file as the bridge
    chunk_tech = (
        chunks_df[["chunk_id", "source_file"]]
        .merge(
            metadata_df.rename(columns={"filename": "source_file", "technology": "technology_meta"}),
            on="source_file", how="left"
        )[["chunk_id", "technology_meta"]]
    )
    out = concerns_df.drop(columns=["technology_meta"], errors="ignore").copy()
    out = out.merge(chunk_tech, on="chunk_id", how="left")
    return out


def recompute_cluster_entropy(concerns_df, n_clusters):
    """Per-cluster normalised technology-entropy, matching 01a's convention.
    Returns dict {cluster_id: normalised_entropy in [0, 1]}."""
    techs = sorted(concerns_df["technology_meta"].dropna().unique())
    tech_to_idx = {t: i for i, t in enumerate(techs)}
    log_n = np.log(max(len(techs), 2))
    result = {}
    for cid, group in concerns_df.groupby("cluster_id"):
        counts = np.zeros(len(techs))
        for t in group["technology_meta"].dropna():
            counts[tech_to_idx[t]] += 1
        if counts.sum() == 0:
            result[int(cid)] = 0.0
        else:
            p = counts / counts.sum()
            result[int(cid)] = float(_entropy(p) / log_n)
    return result


## Run the full multiverse under both metadata variants

Uses the full 450-config grid from `06_multiverse.ipynb`. The only change is that for each configuration, `concerns_df` is relabelled according to the chosen metadata version before the R1/R2/R3 metric functions run.

Each variant takes ~1 minute (the heavy lifting — artefact bundles — is cached per (k, prompt)).

In [ ]:
# @title Build the config grid (same as 06) and run under both metadata variants
DIMENSIONS_V0 = {
    "k":            [60, 75, 90],
    "prompt":       ["V0"],
    "lens_run_idx": [1, 2, 3, 4, 5],
    "sigma2_rule":  ["min_emd", "min_emd_support_cap", "median"],
    "baseline":     ["tech_weighted", "doc_weighted"],
    "threshold":    [0.4, 0.5, 0.6],
}
DIMENSIONS_V3 = {
    "k":            [75],
    "prompt":       ["C_no_decon", "B_broader"],
    "lens_run_idx": [1, 2, 3, 4, 5],
    "sigma2_rule":  ["min_emd", "min_emd_support_cap", "median"],
    "baseline":     ["tech_weighted", "doc_weighted"],
    "threshold":    [0.4, 0.5, 0.6],
}
configs = (
    mv.enumerate_configurations(DIMENSIONS_V0)
    + mv.enumerate_configurations(DIMENSIONS_V3)
)
print(f"Running {len(configs)} configurations under each metadata variant...")

def run_variant(metadata_df, variant_name):
    bundle_cache = {}
    rows = []
    for cfg in configs:
        key = (cfg.k, cfg.prompt)
        if key not in bundle_cache:
            base_bundle = mv.load_pipeline_artifacts(
                cfg.k, cfg.prompt,
                output_folder=OUTPUT_FOLDER,
                checkpoint_folder=CHECKPOINT_FOLDER,
                load_artifacts_fn=_access.load_artifacts,
            )
            b = _copy.copy(base_bundle)
            b["concerns_df"] = relabel_concerns(base_bundle["concerns_df"], metadata_df, chunks_df)
            b["cluster_entropy"] = recompute_cluster_entropy(b["concerns_df"], cfg.k)
            bundle_cache[key] = b
        row = mv.run_configuration(cfg, bundle_cache[key], _address.compute_mixture_weights)
        row["metadata_variant"] = variant_name
        rows.append(row)
    return pd.DataFrame(rows)

orig_df = run_variant(ORIGINAL_META, "original")
alt_df  = run_variant(ALT_META, "data_split")
both_df = pd.concat([orig_df, alt_df], ignore_index=True)
both_df.to_csv(OUTPUT_FOLDER / "metadata_sensitivity_results.csv", index=False)
print(f"Wrote metadata_sensitivity_results.csv ({len(both_df)} rows)")

## Headline comparison across metadata variants

In [ ]:
# @title Side-by-side medians, IQRs and shares
summary = (
    both_df.groupby("metadata_variant")[["ell_R1", "ell_R2", "ell_R3"]]
    .agg(["median", lambda s: s.quantile(0.25), lambda s: s.quantile(0.75), "min", "max"])
    .round(3)
)
summary.columns = pd.MultiIndex.from_tuples([(m, s) for (m, s) in [
    ("ell_R1", "median"), ("ell_R1", "q25"), ("ell_R1", "q75"), ("ell_R1", "min"), ("ell_R1", "max"),
    ("ell_R2", "median"), ("ell_R2", "q25"), ("ell_R2", "q75"), ("ell_R2", "min"), ("ell_R2", "max"),
    ("ell_R3", "median"), ("ell_R3", "q25"), ("ell_R3", "q75"), ("ell_R3", "min"), ("ell_R3", "max"),
]])
print(summary.to_string())

## R2 at the lens level — where does the Privacy & Data Security gap go?

The multiverse metric for R2 is `max pp gap across lenses`. For interpretation, it helps to see the per-lens AI-vs-non-AI pp gap under each metadata variant, specifically whether Privacy & Data Security's dominance shrinks after the data-reclassification.

This is the equivalent of the lens-level table in `03_ai_distinctiveness.ipynb`, computed under both metadata versions and reported side by side.

In [ ]:
# @title Per-lens AI vs non-AI pp gap (canonical config: k=75, V0, lens_run_idx=1, doc_weighted)
# Load the canonical artefacts
_canonical_bundle = mv.load_pipeline_artifacts(
    75, "V0",
    output_folder=OUTPUT_FOLDER,
    checkpoint_folder=CHECKPOINT_FOLDER,
    load_artifacts_fn=_access.load_artifacts,
)
_mapping = _canonical_bundle["per_run_mappings"][0]
_centroids = _canonical_bundle["centroids"]

# Pick a sigma2 that matches the min_emd rule
_sigma2 = mv.SIGMA2_RULES["min_emd"](_canonical_bundle["stability_curve"])
_mix = _address.compute_mixture_weights(_centroids, _mapping, _sigma2, prior="uniform")
mixture_weights = _mix["mixture_weights"]
lens_names = _mix["lens_names"]

def per_lens_gap(concerns_df):
    """Document-weighted AI vs non-AI share per lens."""
    affinities = mixture_weights[concerns_df["cluster_id"].values]  # (n_phrases, n_lenses)
    is_ai = (concerns_df["technology_meta"] == "AI").values
    is_nonai = ~is_ai
    ai_share = affinities[is_ai].mean(axis=0) if is_ai.any() else np.zeros(affinities.shape[1])
    nonai_share = affinities[is_nonai].mean(axis=0) if is_nonai.any() else np.zeros(affinities.shape[1])
    return pd.DataFrame({
        "lens": lens_names,
        "ai_share": ai_share,
        "non_ai_share": nonai_share,
        "ai_minus_nonai_pp": (ai_share - nonai_share) * 100,
    }).sort_values("ai_minus_nonai_pp", ascending=False)

orig_lens = per_lens_gap(relabel_concerns(_canonical_bundle["concerns_df"], ORIGINAL_META, chunks_df))
alt_lens  = per_lens_gap(relabel_concerns(_canonical_bundle["concerns_df"], ALT_META, chunks_df))

lens_comp = orig_lens.rename(columns={"ai_minus_nonai_pp": "gap_original"})[["lens", "gap_original"]].merge(
    alt_lens.rename(columns={"ai_minus_nonai_pp": "gap_data_split"})[["lens", "gap_data_split"]],
    on="lens",
)
lens_comp["delta_pp"] = lens_comp["gap_data_split"] - lens_comp["gap_original"]
lens_comp = lens_comp.sort_values("gap_original", ascending=False)
lens_comp.to_csv(OUTPUT_FOLDER / "metadata_sensitivity_lens_gaps.csv", index=False)

print("Per-lens AI vs non-AI pp gap under each metadata variant (canonical config):")
print(lens_comp.round(2).to_string(index=False))

## Interpretation

Reading guide for the output above:

- **If `gap_data_split` for Privacy & Data Security is much smaller** than `gap_original`, the R2 finding was partly driven by the labelling of data-focused reports as AI. Report this honestly in the paper: the AI-vs-non-AI Privacy gap depends on whether data-governance reports are counted as AI dialogue.
- **If the two columns are close**, the finding is robust to the reclassification — AI dialogue is distinctively concerned with privacy and data security even when data-focused reports are moved to their own category.
- **If other lenses move substantially**, note those too. In particular, Reliability and Safety, Public Understanding, and Governance might shift because the AI subset becomes more model-and-decision-focused once pure data-stewardship reports are removed.

The full 450-config multiverse comparison above (`ell_R2` median under original vs data_split) gives the robust distribution-level answer; the per-lens table gives the interpretable canonical-config answer that can go straight into a paper paragraph.